# Preprocessing Choices and Fairness: A Controlled, Repeated-Split Study on UCI Adult Income

**Extension of P1–P4, informed by:** Valentim, Lourenço & Antunes, *The Impact of Data Preparation on the Fairness of Software Systems* (2019).

## 1. Motivation and research questions
Before a model sees data we scale numbers and encode categories, and sometimes drop the sensitive attribute or resample the training rows. My earlier project (P1–P4) compared four preprocessing pipelines on **one** train/test split, so I could not tell whether small differences were real. This project keeps the idea but makes the comparison more defensible: **five pipelines, the same 30 repeated 70/30 splits (seeds 0–29), paired comparisons with uncertainty.**

* **RQ-A (scaling, my own axis):** Does swapping Standard for Min-Max scaling change performance or fairness?
* **RQ-B (encoding):** Does ordinal encoding (instead of one-hot) change performance or fairness for Logistic Regression?
* **RQ-C (sensitive attribute, paper's RQ1):** Does giving the model `sex` as an input change fairness compared with excluding it?
* **RQ-D (instance selection, paper's RQ3 idea):** Does balancing the four income×sex groups in the *training* data change predictions on the untouched test set?

## 2. Relationship to the 2019 paper
The paper used Decision Trees / Random Forests (Spark), Adult and German Credit, 5-fold CV × 30 seeds, and the metrics CVS (statistical parity difference), DI and NPI. It reported, among other things, that removing the sensitive attribute usually helps fairness but never eliminates unfairness, and that undersampling on label × sensitive attribute made the *training data* perfectly fair while predictions stayed unfair.

**What this notebook does NOT claim:** it is *not* a replication. It uses Logistic Regression (not trees), studies scaling (not in the paper), adds TPR/FPR/equalized-odds gaps, and uses repeated hold-out splits instead of 5-fold CV. The paper's "integer encoding" fed tree algorithms with categorical support; ordinal codes in Logistic Regression are a different experiment (see Section 5).

## 3. The five pipelines
Common to all: UCI Adult (all 48,842 rows), target `income > 50K`, `LogisticRegression(max_iter=2000)`, median imputation for numbers, `"Unknown"` for missing categories, imputers/scalers/encoders fitted on **training data only**. `sex` is always available for **evaluation**; it is a model input **only in F4**.

| ID | Name | Numeric scaler | Categorical encoding | `sex` as model input? | Training rows | Difference from F1 |
|---|---|---|---|---|---|---|
| F1 | Baseline | Standard | One-hot | No | all | — |
| F2 | MinMax | MinMax | One-hot | No | all | scaler only |
| F3 | Ordinal | Standard | Ordinal (+ StandardScaler on codes) | No | all | encoding (and scaling of the codes, as in P4) |
| F4 | Sex-as-input | Standard | One-hot | **Yes** | all | adds `sex` feature |
| F5 | Multivariate undersampling | Standard | One-hot | No | undersampled to equal label×sex cells | training rows only |

## 4. Metrics and sign conventions
Groups: **Male = privileged, Female = unprivileged** (same as the paper's convention). Positive outcome = predicted income > 50K. Let `SR_g = P(Ŷ=1 | group g)`.

* **SPD_signed** = `SR_Male − SR_Female` (privileged minus unprotected; same sign as the paper's CVS). **Positive = favours Males.**
* **DPD_abs** = `|SPD_signed|` (what P1–P4 called Demographic Parity Difference). Always ≥ 0, so it carries **no direction**.
* **DIR** (disparate impact ratio) = `SR_Female / SR_Male` (protected ÷ privileged, same direction as the paper's DI). 1 = parity; below 0.8 fails the "80% rule". I do **not** use Fairlearn's min/max ratio.
* **TPR_diff / FPR_diff** = absolute Male–Female gap in true / false positive rate. **EOD** = max(TPR_diff, FPR_diff).
* **NPI** = normalised mutual information between Ŷ and sex, `MI / sqrt(H(Ŷ)·H(S))` (the paper's definition, computed from the 2×2 table). 0 is best.
* Performance: Accuracy, Precision, Recall, **F1 (primary, since only ~24% are positive)**.

Fairness here means *statistical* group metrics on one dataset; they can disagree with each other.

## 5. Two design caveats (read before interpreting results)
**Ordinal encoding with Logistic Regression (F3).** `OrdinalEncoder` turns each category into one integer (alphabetical order here). A linear model then treats the integer as a quantity: category 5 is "more" than category 2, and the effect is forced to be *linear* in that arbitrary order. One-hot gives each category its own weight. So F3 mixes a representation change with an artificial order, and it also changes the number of input columns. F3 is **not** the paper's integer-encoding experiment (that was for tree models that treat integer-coded columns as categorical).

**F5 changes two things at once.** Undersampling to equal label×sex cells changes (i) the **sex distribution** and (ii) the **income-label distribution** (the training positive rate becomes 50%). Shifting the positive rate also shifts a Logistic Regression's effective decision threshold. So F5 results **cannot** isolate the effect of group balance from the effect of class balance. Only the training set is resampled; the test set is never touched.

In [ ]:
# ---- 1. Setup ----
import importlib, subprocess, sys, warnings, os, time

def ensure(module, pip_name=None):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module])
ensure("ucimlrepo")

import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from ucimlrepo import fetch_ucirepo
warnings.filterwarnings("ignore")

SEEDS = list(range(30))          # 30 repeated 70/30 splits, seeds 0-29
TEST_SIZE = 0.30
N_BOOT = 2000                    # bootstrap resamples for paired-difference intervals
PRIV, UNPRIV = "Male", "Female"
NUM_COLS = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
CAT_COLS = ["workclass", "education", "marital-status", "occupation", "relationship", "race", "native-country"]
OUT_DIR = "results"; os.makedirs(OUT_DIR, exist_ok=True)
print(f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

## 6. Load and clean the data
`?` becomes a real missing value. Nothing is imputed here; imputation happens inside each pipeline so it is fitted on training data only.

In [ ]:
adult = fetch_ucirepo(id=2)
X_all = adult.data.features.copy()
y_raw = adult.data.targets.iloc[:, 0].astype(str).str.strip().str.rstrip(".")

def clean(v):
    if isinstance(v, str):
        v = v.strip()
        return np.nan if v in ("?", "") else v
    return v
for col in CAT_COLS + ["sex"]:
    X_all[col] = X_all[col].astype(object).map(clean)

y = (y_raw == ">50K").astype(int).to_numpy()          # 1 = >50K
sex = X_all["sex"].astype(str).str.strip().to_numpy()  # kept for evaluation
assert set(np.unique(sex)) == {PRIV, UNPRIV}, "unexpected sex values"
strat = pd.Series(y).astype(str).to_numpy() + "_" + sex   # stratify by label x sex
n = len(y)

print("Rows x features:", X_all.shape, "| positive rate:", round(y.mean(), 3))
print("Missing values:", X_all.isna().sum()[X_all.isna().sum() > 0].to_dict())
print("Positive rate by sex (raw data):", {g: round(float(y[sex == g].mean()), 3) for g in (PRIV, UNPRIV)})

## 7. Pipelines, undersampling and metrics (reusable functions)

In [ ]:
PIPELINES = [
    dict(id="F1", name="Baseline",                 scaler="standard", encoding="onehot",  use_sex=False, undersample=False),
    dict(id="F2", name="MinMax",                   scaler="minmax",   encoding="onehot",  use_sex=False, undersample=False),
    dict(id="F3", name="Ordinal",                  scaler="standard", encoding="ordinal", use_sex=False, undersample=False),
    dict(id="F4", name="Sex-as-input",             scaler="standard", encoding="onehot",  use_sex=True,  undersample=False),
    dict(id="F5", name="Multivariate undersampling", scaler="standard", encoding="onehot", use_sex=False, undersample=True),
]
assert len(PIPELINES) == 5
PIDS = [p["id"] for p in PIPELINES]
SCALERS = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}

def feature_cols(p):
    return NUM_COLS + CAT_COLS + (["sex"] if p["use_sex"] else [])

def build_model(p, seed):
    """Imputation + scaling + encoding + classifier in ONE Pipeline: fit() sees training data only."""
    cat_cols = CAT_COLS + (["sex"] if p["use_sex"] else [])
    num = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", SCALERS[p["scaler"]]())])
    imp = SimpleImputer(strategy="constant", fill_value="Unknown")
    if p["encoding"] == "onehot":
        cat = Pipeline([("imp", imp), ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    else:  # ordinal (+ StandardScaler on the codes, as in my P4)
        cat = Pipeline([("imp", imp),
                        ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                        ("sc", StandardScaler())])
    prep = ColumnTransformer([("num", num, NUM_COLS), ("cat", cat, cat_cols)], sparse_threshold=0)
    return Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, random_state=seed))])

def undersample_multivariate(y_tr, s_tr, rng):
    """Randomly drop rows so all four (label x sex) cells have the size of the smallest cell. Training data only."""
    cell = np.array([f"{a}_{b}" for a, b in zip(y_tr, s_tr)])
    labels, counts = np.unique(cell, return_counts=True)
    n_min = counts.min()
    keep = np.concatenate([rng.choice(np.flatnonzero(cell == c), n_min, replace=False) for c in labels])
    rng.shuffle(keep)
    return keep

def _entropy(q):
    q = q[q > 0]; return float(-(q * np.log(q)).sum())

def npi(pred, s):
    f = (s == UNPRIV).astype(int)
    joint = np.array([[np.sum((pred == a) & (f == b)) for b in (0, 1)] for a in (0, 1)]) / len(pred)
    pa, pb = joint.sum(1), joint.sum(0)
    mi = sum(joint[i, j] * np.log(joint[i, j] / (pa[i] * pb[j])) for i in (0, 1) for j in (0, 1) if joint[i, j] > 0)
    d = np.sqrt(_entropy(pa) * _entropy(pb))
    return mi / d if d > 0 else np.nan

def rate(mask, values):
    return float(values[mask].mean()) if mask.sum() > 0 else np.nan

def all_metrics(y_true, pred, s):
    m, f = s == PRIV, s == UNPRIV
    sr_m, sr_f = rate(m, pred), rate(f, pred)
    tpr_m, tpr_f = rate(m & (y_true == 1), pred), rate(f & (y_true == 1), pred)
    fpr_m, fpr_f = rate(m & (y_true == 0), pred), rate(f & (y_true == 0), pred)
    tpr_d, fpr_d = abs(tpr_m - tpr_f), abs(fpr_m - fpr_f)
    return {"Accuracy": accuracy_score(y_true, pred),
            "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred, zero_division=0),
            "F1": f1_score(y_true, pred, zero_division=0),
            "SelRate_Male": sr_m, "SelRate_Female": sr_f,
            "SPD_signed": sr_m - sr_f,                      # Male minus Female (positive = favours Males)
            "DPD_abs": abs(sr_m - sr_f),
            "DIR": sr_f / sr_m if sr_m > 0 else np.nan,     # Female / Male
            "TPR_diff": tpr_d, "FPR_diff": fpr_d, "EOD": max(tpr_d, fpr_d),
            "NPI": npi(pred, s)}

def run_split(seed, pipelines=PIPELINES):
    tr, te = train_test_split(np.arange(n), test_size=TEST_SIZE, random_state=seed, stratify=strat)
    rng = np.random.default_rng(seed)
    rows = []
    for p in pipelines:
        tr_idx = tr[undersample_multivariate(y[tr], sex[tr], rng)] if p["undersample"] else tr
        cols = feature_cols(p)
        model = build_model(p, seed).fit(X_all.iloc[tr_idx][cols], y[tr_idx])      # training rows only
        pred = model.predict(X_all.iloc[te][cols])                                  # test rows: transform + predict only
        ytr, str_ = y[tr_idx], sex[tr_idx]
        row = {"seed": seed, "ID": p["id"], "Pipeline": f'{p["id"]} {p["name"]}',
               "n_train": len(tr_idx), "train_pos_rate": float(ytr.mean()),
               "train_female_share": float((str_ == UNPRIV).mean()),
               "train_label_SPD": rate(str_ == PRIV, ytr) - rate(str_ == UNPRIV, ytr)}
        row.update(all_metrics(y[te], pred, sex[te]))
        rows.append(row)
    return rows

## 8. Sanity check against my earlier P1–P4 notebook (seed 42)
Same split rule and seed as before, so F1, F2 and F3 should reproduce the earlier P1, P2 and P4 numbers (up to library-version effects). This is only a consistency check, not part of the five-pipeline analysis.

In [ ]:
# Reference values copied from my earlier P1-P4 results (single split, random_state=42)
REF = {"F1": dict(Accuracy=0.8525, F1=0.6612, DPD_abs=0.1785),
       "F2": dict(Accuracy=0.8487, F1=0.6505, DPD_abs=0.1819),
       "F3": dict(Accuracy=0.8198, F1=0.5348, DPD_abs=0.1407)}
chk = pd.DataFrame(run_split(42, [p for p in PIPELINES if p["id"] in REF])).set_index("ID")
rows = []
for pid, ref in REF.items():
    for m, v in ref.items():
        rows.append({"ID": pid, "metric": m, "earlier": v, "now": round(chk.loc[pid, m], 4),
                     "match(|diff|<=0.001)": abs(chk.loc[pid, m] - v) <= 0.001})
display(pd.DataFrame(rows))

## 9. Run all five pipelines on 30 repeated splits
Every pipeline uses the **same** train/test split in each seed, so comparisons are paired. Expected runtime on free Colab CPU: a few minutes.

In [ ]:
t0 = time.time(); all_rows = []
for seed in SEEDS:
    all_rows += run_split(seed)
    if (seed + 1) % 10 == 0:
        print(f"finished {seed + 1}/{len(SEEDS)} splits ({time.time() - t0:.0f}s)")
df = pd.DataFrame(all_rows)
assert len(df) == 5 * len(SEEDS) and df["ID"].nunique() == 5
df.to_csv(f"{OUT_DIR}/results_per_split.csv", index=False)
print("rows:", len(df), "| NaN metrics:", int(df.isna().sum().sum()))

## 10. Final results table (mean ± std over 30 splits)
Std is the spread across splits (not a confidence interval of the mean).

In [ ]:
PERF = ["Accuracy", "Precision", "Recall", "F1"]
FAIR = ["SPD_signed", "DPD_abs", "DIR", "TPR_diff", "FPR_diff", "EOD", "NPI"]
METRICS = PERF + ["SelRate_Male", "SelRate_Female"] + FAIR
g = df.groupby("ID")
mean_df, std_df = g[METRICS].mean().loc[PIDS], g[METRICS].std().loc[PIDS]
names = df.drop_duplicates("ID").set_index("ID")["Pipeline"]
summary = pd.concat([mean_df.add_suffix("_mean"), std_df.add_suffix("_std")], axis=1)
summary.insert(0, "Pipeline", names.loc[PIDS])
summary["DIR_below_0.8_share"] = g["DIR"].apply(lambda s: (s < 0.8).mean()).loc[PIDS]
summary.to_csv(f"{OUT_DIR}/summary_mean_std.csv")
pretty = pd.DataFrame({"Pipeline": names.loc[PIDS]})
for m in METRICS:
    pretty[m] = [f"{mean_df.loc[i, m]:.4f} ± {std_df.loc[i, m]:.4f}" for i in PIDS]
pretty["DIR<0.8 in splits"] = [f"{summary.loc[i, 'DIR_below_0.8_share']*30:.0f}/30" for i in PIDS]
display(pretty[["Pipeline"] + PERF + ["DIR<0.8 in splits"]])
display(pretty[["Pipeline", "SelRate_Male", "SelRate_Female"] + FAIR])

### Training-data statistics (shows exactly what F4/F5 change)
F5 changes both the sex mix and the label mix of the training set; `train_label_SPD` is the data-level Male−Female gap in the training labels.

In [ ]:
tstats = g[["n_train", "train_pos_rate", "train_female_share", "train_label_SPD"]].mean().loc[PIDS].round(4)
tstats.to_csv(f"{OUT_DIR}/train_data_stats.csv"); display(tstats)

## 11. Paired comparisons against F1
For each seed: `difference = metric(Fk) − metric(F1)` on the **same** split. We report the mean difference, a 95% percentile bootstrap interval of that mean, and how often the difference is positive. The 30 splits overlap in their data, so these intervals are somewhat optimistic; treat them as a guide, not a formal test.

In [ ]:
rng_b = np.random.default_rng(0)
base = df[df.ID == "F1"].set_index("seed")
paired = []
for pid in PIDS[1:]:
    other = df[df.ID == pid].set_index("seed")
    for m in METRICS:
        d = (other[m] - base[m]).dropna().to_numpy()
        boots = rng_b.choice(d, size=(N_BOOT, len(d)), replace=True).mean(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        paired.append({"ID": pid, "metric": m, "mean_diff": d.mean(), "ci_low": lo, "ci_high": hi,
                       "std_diff": d.std(ddof=1), "share_positive": (d > 0).mean(), "ci_excludes_0": (lo > 0) or (hi < 0)})
paired = pd.DataFrame(paired); paired.to_csv(f"{OUT_DIR}/paired_differences_vs_F1.csv", index=False)
KEY = ["F1", "DPD_abs", "DIR", "TPR_diff", "FPR_diff", "EOD"]
view = paired[paired.metric.isin(KEY)].copy()
view["diff [95% CI]"] = view.apply(lambda r: f"{r.mean_diff:+.4f} [{r.ci_low:+.4f}, {r.ci_high:+.4f}]", axis=1)
display(view.pivot(index="ID", columns="metric", values="diff [95% CI]")[KEY])
display(view.pivot(index="ID", columns="metric", values="ci_excludes_0")[KEY].rename_axis("CI excludes 0?"))

## 12. Plots (all drawn from the computed results)

In [ ]:
COL = dict(zip(PIDS, plt.cm.tab10.colors[:5]))
def finish(fig, fname):
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{fname}", dpi=120); plt.show()

# 12.1 Mean +/- std per pipeline
fig, axs = plt.subplots(2, 3, figsize=(14, 7))
for ax, m in zip(axs.ravel(), KEY):
    ax.bar(PIDS, mean_df.loc[PIDS, m], yerr=std_df.loc[PIDS, m], capsize=4, color=[COL[i] for i in PIDS])
    ax.set_title(m + (" (higher = better)" if m == "F1" else " (ideal = 1; <0.8 fails 80% rule)" if m == "DIR" else " (0 = no gap)"), fontsize=10)
    if m == "DIR": ax.axhline(0.8, color="red", ls="--", lw=1)
    ax.grid(axis="y", alpha=.3)
fig.suptitle("Mean ± std over 30 splits"); finish(fig, "fig1_mean_std.png")

In [ ]:
# 12.2 Paired differences vs F1 (forest plot)
fig, axs = plt.subplots(1, len(KEY), figsize=(17, 3.6), sharey=True)
for ax, m in zip(axs, KEY):
    sub = paired[paired.metric == m].set_index("ID").loc[PIDS[1:]]
    ypos = np.arange(len(sub))
    ax.errorbar(sub.mean_diff, ypos, xerr=[sub.mean_diff - sub.ci_low, sub.ci_high - sub.mean_diff], fmt="o", capsize=4, color="k")
    ax.axvline(0, color="grey", ls="--"); ax.set_yticks(ypos); ax.set_yticklabels(sub.index); ax.set_title(f"Δ {m}", fontsize=10); ax.grid(alpha=.3)
fig.suptitle("Paired difference vs F1 (mean and 95% bootstrap interval; line at 0 = no difference)", y=1.03); finish(fig, "fig2_paired_forest.png")

# 12.3 Accuracy/F1 vs fairness gap
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, fm in zip(axs, ["DPD_abs", "EOD"]):
    for i in PIDS:
        ax.errorbar(mean_df.loc[i, "F1"], mean_df.loc[i, fm], xerr=std_df.loc[i, "F1"], yerr=std_df.loc[i, fm],
                    fmt="o", color=COL[i], capsize=3, label=i)
    ax.set_xlabel("F1 (higher = better)"); ax.set_ylabel(f"{fm} (lower = smaller gap)"); ax.grid(alpha=.3); ax.legend()
fig.suptitle("Performance vs fairness gap (mean ± std)"); finish(fig, "fig3_tradeoff.png")

In [ ]:
# 12.4 Spread over seeds
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, m in zip(axs, ["DPD_abs", "EOD", "DIR"]):
    ax.boxplot([df.loc[df.ID == i, m] for i in PIDS]); ax.set_xticklabels(PIDS)
    if m == "DIR": ax.axhline(0.8, color="red", ls="--", lw=1)
    ax.set_title(m); ax.grid(axis="y", alpha=.3)
fig.suptitle("Distribution over 30 splits"); finish(fig, "fig4_boxplots.png")

# 12.5 Selection rate by sex
fig, ax = plt.subplots(figsize=(8, 4.5)); x = np.arange(len(PIDS)); w = 0.38
ax.bar(x - w/2, mean_df.loc[PIDS, "SelRate_Male"], w, label="Male"); ax.bar(x + w/2, mean_df.loc[PIDS, "SelRate_Female"], w, label="Female")
ax.set_xticks(x); ax.set_xticklabels(PIDS); ax.set_ylabel("P(predicted >50K)"); ax.set_title("Mean selection rate by sex"); ax.legend(); ax.grid(axis="y", alpha=.3)
finish(fig, "fig5_selection_rates.png")

## 13. Results and conclusions (generated from the computed numbers)
The text below is produced by code from this run's results. It describes differences and whether the interval excludes 0; it does not explain causes.

In [ ]:
def pr(pid, m):
    r = paired[(paired.ID == pid) & (paired.metric == m)].iloc[0]
    s = "interval EXCLUDES 0" if r.ci_excludes_0 else "interval includes 0 (no clear difference)"
    return f"{m}: {r.mean_diff:+.4f} [{r.ci_low:+.4f}, {r.ci_high:+.4f}] -> {s}"
LABEL = {"F2": "RQ-A scaling (MinMax vs Standard)", "F3": "RQ-B encoding (Ordinal vs One-hot)",
         "F4": "RQ-C sex as input (vs excluded)", "F5": "RQ-D multivariate undersampling (vs all training rows)"}
print("PAIRED DIFFERENCES vs F1 (mean over 30 splits, 95% bootstrap interval)\n")
for pid in PIDS[1:]:
    print(f"{pid} - {LABEL[pid]}")
    for m in KEY: print("   ", pr(pid, m))
print("\nOVERALL")
best_f1 = mean_df["F1"].idxmax(); small_dpd = mean_df["DPD_abs"].idxmin(); small_eod = mean_df["EOD"].idxmin()
print(f"- Highest mean F1: {best_f1}; smallest mean DPD_abs: {small_dpd}; smallest mean EOD: {small_eod}.")
print("- Share of splits with DIR < 0.8 (fails 80% rule): " + ", ".join(f"{i}={summary.loc[i,'DIR_below_0.8_share']:.0%}" for i in PIDS))
print("- Males predicted >50K more often than Females (mean SelRate): " + ", ".join(f"{i}={'yes' if mean_df.loc[i,'SPD_signed']>0 else 'no'}" for i in PIDS))
print("- Largest mean DPD_abs vs smallest across pipelines differ by "
      f"{mean_df['DPD_abs'].max() - mean_df['DPD_abs'].min():.4f}; F1 differs by {mean_df['F1'].max() - mean_df['F1'].min():.4f}.")
print("\nCAUTIONS: one dataset, one classifier, one protected attribute; overlapping hold-out splits make intervals optimistic; "
      "F3 mixes encoding with arbitrary category order; F5 changes sex mix and label mix together, so its effect cannot be attributed to either alone.")

## 14. Limitations and threats to validity
* **Scope:** one dataset (Adult, 1994 census), one classifier (Logistic Regression, default `C`, 0.5 threshold), one protected attribute (`sex`, binary). Results need not carry over to trees, other datasets or other attributes, and **they are not a replication of the paper**.
* **Statistics:** the 30 hold-out splits overlap, so they are not independent; bootstrap intervals are indicative only. No multiple-comparison correction is applied.
* **Proxies:** `relationship`, `marital-status`, `occupation` can encode sex; excluding `sex` (F1–F3, F5) does not remove that information. `fnlwgt` (a census sampling weight) and `race` remain model features for continuity with P1–P4.
* **F3:** ordinal codes impose an arbitrary alphabetical order on a linear model, so its differences cannot be pinned on "encoding" alone.
* **F5:** changes the sex distribution and the label distribution simultaneously, and moves the effective decision threshold. A label-only undersampling variant would be needed to separate these, and is not part of the five approved pipelines.
* **Metrics:** group-statistical fairness metrics can disagree; none of them proves a model is or is not "fair". Choosing a missing-value strategy ("Unknown" category) differs from the paper's.
* **Dropped from P1–P4:** the Robust-scaling pipeline (P3) is not part of this study. `RobustScaler` has zero IQR on `capital-gain` and `capital-loss`, which makes it a degenerate scaling contrast.

## 15. Conclusion
The conclusions are the printed output of Section 13 plus the tables and figures above, so they always match the actual run. In the report, state each finding with its size and interval, restrict it to this dataset and classifier, and avoid saying that any step *caused* a fairness change or that any pipeline is "fair" or "fairest" without the paired intervals.

In [ ]:
# ---- Export: zip all CSVs and figures ----
import shutil
shutil.make_archive("fairness_results", "zip", OUT_DIR)
print("Saved:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download("fairness_results.zip")
except Exception:
    print("Not on Colab: find fairness_results.zip in the working folder.")

## 16. How to run and download
1. Open this file in Google Colab (File → Upload notebook). The free **CPU** runtime is enough; no GPU needed.
2. Click **Runtime → Run all**. The dataset downloads automatically through `ucimlrepo` (internet required). The 150 model fits take a few minutes.
3. The last code cell zips everything into `fairness_results.zip` and starts the download. If the browser blocks it, open the **Files** panel (folder icon, left side) and download `fairness_results.zip` or the `results/` folder.
4. Contents: `results_per_split.csv` (every seed × pipeline), `summary_mean_std.csv`, `paired_differences_vs_F1.csv`, `train_data_stats.csv` and five `.png` figures.
5. Results are reproducible: seeds 0–29 are fixed and library versions are printed at the top. Small numeric differences may appear with different scikit-learn versions.